In [1]:
from transformers import AutoProcessor, AutoModelForCausalLM
import torch


model_path = "model/florence2"
model = (
    AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        # load_in_8bit=True,
        trust_remote_code=True,
    )
    .cuda()
    .eval()
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)


/home/zaln/miniconda3/envs/magiv3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zaln/miniconda3/envs/magiv3/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Florence2LanguageForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warn

In [ ]:
# 卸载模型
del model
del processor
torch.cuda.empty_cache()

In [ ]:

img_paths = ["/home/zaln/文档/AAvscode/magiv3/input/1_en.jpg"]


In [ ]:
# 原始调用（单张）

from PIL import Image
from model.florence2.utils import visualise_single_image_prediction


img = Image.open(img_paths[0]).convert("RGB")
result = model.predict_detections_and_associations([img], processor)[0]
result["dialog_confidences"] = [1.0] * len(result["texts"])

# import numpy as np

# visualise_single_image_prediction(
#     np.array(img), result, "output/" + "original_" + img_paths[0].split("/")[-1]
# )
print(result.keys())
print(result)

In [ ]:
# ocr_utils reload

import ocr_utils
import importlib

importlib.reload(ocr_utils)

# OCR - paddleocr

In [ ]:
# 注入后调用（多张）

from ocr_utils import get_ocr_results, predict_with_injected_ocr
from model.florence2.utils import visualise_single_image_prediction

unordered_ocr_res = get_ocr_results(img_paths, only_white_bg=False, zh_texts=True)
results = predict_with_injected_ocr(model, processor, img_paths, unordered_ocr_res)

# 绘制

import numpy as np
from PIL import Image
from ocr_utils import print_texts

img = Image.open(img_paths[0]).convert("RGB")
visualise_single_image_prediction(
    np.array(img), results[0], "output/" + "injected_" + img_paths[0].split("/")[-1]
)
print_texts(img_paths[0], results[0]["ocr_texts"])

In [ ]:
for i, panel in enumerate(results[0]["panels"]):
    img.crop(panel).save(f"input/panels/panel{i}_en.jpg")


In [ ]:
print(results[0].keys())

results

In [ ]:
# fake

results = [
    {
        "panels": [
            [
                395.03997802734375,
                0.7055000066757202,
                919.199951171875,
                487.5005187988281,
            ],
            [
                0.47999998927116394,
                65.61150360107422,
                393.1199951171875,
                487.5005187988281,
            ],
            [448.79998779296875, 505.843505859375, 919.199951171875, 851.5385131835938],
            [
                0.47999998927116394,
                505.843505859375,
                450.7200012207031,
                851.5385131835938,
            ],
            [
                631.2000122070312,
                864.2374877929688,
                919.199951171875,
                1410.2945556640625,
            ],
            [
                375.8399963378906,
                864.2374877929688,
                629.2799682617188,
                1339.7445068359375,
            ],
            [
                0.47999998927116394,
                860.0045166015625,
                377.7599792480469,
                1410.2945556640625,
            ],
        ],
        "texts": [
            [825.0, 50.0, 885.0, 204.0],
            [720.0, 86.0, 804.0, 250.0],
            [458.0, 73.0, 570.0, 281.0],
            [251.0, 73.0, 344.0, 197.0],
            [54.0, 315.0, 194.0, 456.0],
            [875.0, 520.0, 907.0, 651.0],
            [755.0, 555.0, 864.0, 697.0],
            [476.0, 547.0, 537.0, 689.0],
            [329.0, 531.0, 421.0, 659.0],
            [76.0, 520.0, 199.0, 681.0],
            [68.0, 718.0, 108.0, 827.0],
            [931.0, 1379.0, 954.0, 1402.0],
            [805.0, 894.0, 893.0, 1086.0],
            [641.0, 920.0, 727.0, 1045.0],
            [503.0, 886.0, 612.0, 1048.0],
            [328.0, 898.0, 360.0, 1045.0],
            [88.0, 914.0, 150.0, 1061.0],
        ],
        "characters": [
            [
                503.5199890136719,
                162.9705047607422,
                826.0799560546875,
                481.85650634765625,
            ],
            [20.639999389648438, 85.3655014038086, 252.0, 480.44549560546875],
            [
                270.239990234375,
                148.86050415039062,
                392.1600036621094,
                480.44549560546875,
            ],
            [455.5199890136719, 514.3095092773438, 786.719970703125, 841.6614990234375],
            [204.0, 565.1055297851562, 447.8399963378906, 841.6614990234375],
            [
                723.3599853515625,
                1150.6705322265625,
                918.239990234375,
                1408.883544921875,
            ],
            [633.1199951171875, 1058.95556640625, 715.6799926757812, 1408.883544921875],
            [
                435.3599853515625,
                1092.8194580078125,
                625.4400024414062,
                1328.45654296875,
            ],
            [
                379.67999267578125,
                1097.052490234375,
                495.8399963378906,
                1327.0455322265625,
            ],
            [
                0.47999998927116394,
                860.0045166015625,
                287.5199890136719,
                1408.883544921875,
            ],
        ],
        "tails": [
            [
                711.8399658203125,
                227.87649536132812,
                743.5199584960938,
                260.3294982910156,
            ],
            [
                190.55999755859375,
                178.4915008544922,
                228.95999145507812,
                209.53350830078125,
            ],
            [
                204.95999145507812,
                428.2384948730469,
                255.83999633789062,
                459.280517578125,
            ],
            [717.5999755859375, 639.8884887695312, 760.7999877929688, 682.218505859375],
            [396.0, 677.9855346679688, 427.67999267578125, 721.7265014648438],
            [27.35999870300293, 632.83349609375, 68.63999938964844, 662.4644775390625],
            [128.16000366210938, 799.3314819335938, 174.239990234375, 830.37353515625],
            [
                856.7999877929688,
                1098.4635009765625,
                885.5999755859375,
                1142.2044677734375,
            ],
            [
                697.4400024414062,
                1040.612548828125,
                725.2799682617188,
                1081.531494140625,
            ],
            [514.0800170898438, 1057.5445556640625, 540.9599609375, 1094.23046875],
            [
                317.2799987792969,
                1057.5445556640625,
                349.91998291015625,
                1091.4085693359375,
            ],
            [
                94.55999755859375,
                1064.5994873046875,
                122.39999389648438,
                1105.5185546875,
            ],
        ],
        "ocr_texts": [
            "但我认为②是行不通的。",
            "到时咒灵的危害会转变为诅咒师的危害。",
            "但是①也有①的问题搞不好反而会催生出许多真剑那样身体天赋异禀的人。",
            "那些人同样有可能是坏人吧？",
            "除非天生具备极其强大的术式或咒力，否则不会出现那样的人的。",
            "这样的话……",
            "就按照①的消除日本人和希姆利亚人的咒力",
            "来推进应该就没问题了吧？",
            "希姆利亚人不会产生咒灵吧。",
            "若单单消除地球人的咒力日后还会产生矛盾。",
            "……确实",
            "画",
            "话都说到这份上了，可是啊……那个……",
            "抱歉还未做自我介绍我是马鲁鲁。",
            "马鲁鲁你对我们接下来要做的事是什么想法？",
            "……为了赎罪。",
            "我们给地球添了许多麻烦。",
        ],
        "character_cluster_labels": [0, 1, 2, 1, 2, 2, 1, 2, 1, 1],
        "text_character_associations": [
            [0, 0],
            [1, 0],
            [2, 0],
            [3, 1],
            [5, 3],
            [6, 3],
            [7, 3],
            [8, 4],
            [10, 4],
            [12, 5],
            [13, 6],
            [14, 7],
            [15, 9],
            [16, 9],
        ],
        "text_tail_associations": [
            [0, 0],
            [1, 0],
            [3, 1],
            [4, 2],
            [6, 3],
            [8, 4],
            [10, 6],
            [12, 7],
            [13, 8],
            [14, 9],
            [15, 10],
            [16, 11],
        ],
        "is_essential_text": [
            True,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
            True,
            False,
            True,
            True,
            True,
            True,
            True,
        ],
        "dialog_confidences": [
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
        ],
    }
]

img_paths = ["input/1_zh.jpg"]


# caption - llamacpp

In [ ]:
from ocr_utils import get_captions
captions = get_captions(img_paths, results, think=False)


In [ ]:
for idx, caption in enumerate(captions[0]):
    print(f"🍥 panel{idx}:")
    print(caption)
    print("\n"+"-" * 90 + "\n")


# character grounding

In [ ]:
from PIL import Image

# img = Image.open(img_paths[0])
# panels = results[0]["panels"]
# img_panels = [img.crop(panel) for panel in panels]
# grounded_results = model.predict_character_grounding(
#     [img_panels[0]],
#     [captions[0][0]],
#     processor,
#     strict=False,
# )

# # 第 0 张图片的全部 panel ，串行
# img = Image.open(img_paths[0])
# img_panels = [img.crop(panel) for panel in results[0]["panels"]]
# grounded_results = []
# for img, cap in zip(img_panels, captions[0]):
#     result = model.predict_character_grounding(
#         [img],
#         [cap],
#         processor,
#         strict=False,
#     )
#     grounded_results.extend(result)




In [5]:
from PIL import Image
from ocr_utils import visualize_grounding

img = Image.open((f"input/panels/panel4_zh.jpg")).convert("RGB")

caption = "This black and white manga panel depicts a scene between two characters. The character on the left is shown in profile, with long, dark hair styled in a high ponytail and wearing a dark, high-collared garment. Their expression is neutral as they look towards the other figure. The character on the right is seated on the ground with their legs crossed, their face obscured by the deep shadow of a light-colored hoodie. This figure appears to be in a state of distress or exhaustion, slumped forward with their head down. The background is minimal, with simple lines suggesting a wooden floor or platform and a starburst effect in the upper portion of the frame."

result = model.predict_character_grounding([img], [caption], processor, strict=False)
draw_img, tagged_caption = visualize_grounding(img, result[0])
draw_img.show()
print(tagged_caption)

/home/zaln/miniconda3/envs/magiv3/lib/python3.10/site-packages/torch/utils/checkpoint.py:86: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


This black and white manga panel depicts a scene between two characters. The character on the left[0] is shown in profile, with long, dark hair styled in a high ponytail and wearing a dark, high-collared garment. Their[0] expression is neutral as they[0] look towards the other figure. The Character on the right[1] is seated on the ground with their[1] legs crossed, their[1] face obscured by the deep shadow of a light-colored hoodie. This figure[1] appears to be in a state of distress or exhaustion, slumped forward with their head down. The background is minimal, with simple lines suggesting a wooden floor or platform and a starburst effect in the upper portion of the frame.


In [5]:
print(result)

[{'grounded_caption': 'This black and white manga panel depicts a scene between two characters. The character on the left is shown in profile, with long, dark hair styled in a high ponytail and wearing a dark, high-collared garment. Their expression is neutral as they look towards the other figure. The other figure. The Character on the right is seated on the ground with their legs crossed, their face obscured by the deep shadow of a light-colored hoodie. This figure appears to be in a state of distress or exhaustion, slumped forward with their head down. The background is minimal, with simple lines suggesting a wooden floor or platform and a starburst effect in the upper portion of the frame.', 'bboxes': [[[3.7695000171661377, 238.7324981689453, 105.72550201416016, 686.656494140625]], [[3.7695000171661377, 238.7324981689453, 106.80249786376953, 686.656494140625]], [[3.7695000171661377, 238.7324981689453, 106.44349670410156, 686.656494140625]], [[112.54650115966797, 354.14849853515625,

This black and white manga panel depicts a scene between two characters. The character on the left[0] is shown in profile, with long, dark hair styled in a high ponytail and wearing a dark, high-collared garment. Their[0] expression is neutral as they[0] look towards the other figure[1]. The other figure. The Character[1] on the right is seated on the ground with their[1] legs crossed, their[1] face obscured by the deep shadow of a light-colored hoodie. This figure[1] appears to be in a state of distress or exhaustion, slumped forward with their head down. The background is minimal, with simple lines suggesting a wooden floor or platform and a starburst effect in the upper portion of the frame.

In [2]:
from ocr_utils import visualize_grounding
from PIL import Image

for i in range(len(img_panels)):
    draw_img, tagged_caption = visualize_grounding(img_panels[i], grounded_results[i])
    draw_img.save(f"output/panel{i}_grounded.png")
    print(f"🌸 panel{i} caption:")
    print(tagged_caption)
    print("-" * 90)

NameError: name 'img_panels' is not defined